In [ ]:
from collections import deque

# =========================================================
# 8 PUZZLE DÙNG BFS
# Có NODE / FRONTIER / REACHED
# =========================================================

GOAL = (1, 2, 3,
        4, 5, 6,
        7, 8, 0)

SIZE = 3

ACTIONS = {
    "U": (-1, 0),
    "D": (1, 0),
    "L": (0, -1),
    "R": (0, 1)
}


class Node:
    def __init__(self, name, state, parent=None, action=None, depth=0):
        self.name = name
        self.state = state
        self.parent = parent
        self.action = action
        self.depth = depth


def print_board(state):
    print("+---+---+---+")
    for i in range(0, 9, 3):
        row = state[i:i+3]
        text = "|"
        for value in row:
            if value == 0:
                text += "   |"
            else:
                text += f" {value} |"
        print(text)
        print("+---+---+---+")


def short_state(state):
    return "[" + " ".join(str(x) for x in state) + "]"


def is_inside(row, col):
    return 0 <= row < SIZE and 0 <= col < SIZE


def move_state(state, action):
    blank = state.index(0)
    row, col = divmod(blank, SIZE)

    dr, dc = ACTIONS[action]
    new_row = row + dr
    new_col = col + dc

    if not is_inside(new_row, new_col):
        return None

    new_blank = new_row * SIZE + new_col

    temp = list(state)
    temp[blank], temp[new_blank] = temp[new_blank], temp[blank]

    return tuple(temp)


def create_children(parent_node, next_id):
    children = []

    for action in ["U", "D", "L", "R"]:
        new_state = move_state(parent_node.state, action)

        if new_state is not None:
            child = Node(
                name="N" + str(next_id),
                state=new_state,
                parent=parent_node,
                action=action,
                depth=parent_node.depth + 1
            )
            children.append(child)
            next_id += 1

    return children, next_id


def is_solvable(state):
    nums = [x for x in state if x != 0]
    inv = 0

    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] > nums[j]:
                inv += 1

    return inv % 2 == 0


def reconstruct_path(goal_node):
    path = []
    node = goal_node

    while node is not None:
        path.append(node)
        node = node.parent

    path.reverse()
    return path


def print_node_table(node_rows):
    print("NODE TABLE")
    print("-" * 100)
    print(f"{'Step':<7}{'Node':<8}{'Parent':<10}{'Action':<10}{'Depth':<8}{'State'}")
    print("-" * 100)

    for row in node_rows:
        print(
            f"{row['step']:<7}"
            f"{row['node']:<8}"
            f"{row['parent']:<10}"
            f"{row['action']:<10}"
            f"{row['depth']:<8}"
            f"{row['state']}"
        )

    print()


def print_frontier_table(frontier):
    print("FRONTIER TABLE")
    print("-" * 100)
    print(f"{'Order':<8}{'Node':<8}{'Parent':<10}{'Action':<10}{'Depth':<8}{'State'}")
    print("-" * 100)

    if len(frontier) == 0:
        print("Frontier rỗng")
    else:
        for i, node in enumerate(frontier, start=1):
            parent = node.parent.name if node.parent else "None"
            action = node.action if node.action else "None"

            print(
                f"{i:<8}"
                f"{node.name:<8}"
                f"{parent:<10}"
                f"{action:<10}"
                f"{node.depth:<8}"
                f"{short_state(node.state)}"
            )

    print()


def print_reached_table(reached_order):
    print("REACHED TABLE")
    print("-" * 100)
    print(f"{'No':<8}{'State'}")
    print("-" * 100)

    for i, state in enumerate(reached_order, start=1):
        print(f"{i:<8}{short_state(state)}")

    print()


def print_children(parent_node, children, added_names, ignored_states):
    print("CHILDREN GENERATED")
    print("-" * 100)

    if len(children) == 0:
        print("Node này không sinh được node con")
    else:
        for child in children:
            if child.name in added_names:
                status = "ADD TO FRONTIER"
            else:
                status = "IGNORE - ALREADY REACHED"

            print(
                f"({parent_node.name}, {child.action}, 1) -> {child.name} | "
                f"{short_state(child.state)} | {status}"
            )

    if len(ignored_states) > 0:
        print()
        print("State bị bỏ qua vì đã có trong Reached:")
        for state in ignored_states:
            print("-", short_state(state))

    print()


def print_solution(path):
    print("SOLUTION PATH")
    print("-" * 100)

    for i, node in enumerate(path):
        parent = node.parent.name if node.parent else "None"
        action = node.action if node.action else "START"

        print(f"Step {i}: Node {node.name} | Parent: {parent} | Action: {action} | Depth: {node.depth}")
        print_board(node.state)

    actions = [node.action for node in path if node.action is not None]

    print("Chuỗi hành động:", actions)
    print("Số bước giải:", len(actions))
    print()


def print_one_bfs_step(start, goal, current_node, node_rows, children, added_names, ignored_states, frontier, reached_order):
    print("=" * 100)
    print("BFS 8 PUZZLE")
    print("=" * 100)

    print("START STATE")
    print_board(start)

    print("GOAL STATE")
    print_board(goal)

    print("NODE ĐANG XÉT:", current_node.name)
    print_board(current_node.state)

    print_node_table(node_rows)
    print_children(current_node, children, added_names, ignored_states)
    print_frontier_table(list(frontier))
    print_reached_table(reached_order)


def bfs_8_puzzle(start, goal=GOAL, max_steps=100):
    if not is_solvable(start):
        print("Bàn cờ này không giải được.")
        return None

    root = Node("A", start, parent=None, action=None, depth=0)

    frontier = deque([root])
    reached = {root.state}
    reached_order = [root.state]
    node_rows = []

    step = 0
    next_id = 1

    while len(frontier) > 0 and step < max_steps:
        current_node = frontier.popleft()
        step += 1

        node_rows.append({
            "step": step,
            "node": current_node.name,
            "parent": current_node.parent.name if current_node.parent else "None",
            "action": current_node.action if current_node.action else "None",
            "depth": current_node.depth,
            "state": short_state(current_node.state)
        })

        if current_node.state == goal:
            print("=" * 100)
            print("BFS 8 PUZZLE")
            print("=" * 100)
            print("START STATE")
            print_board(start)
            print("GOAL STATE")
            print_board(goal)
            print("NODE ĐANG XÉT:", current_node.name)
            print_board(current_node.state)
            print_node_table(node_rows)
            print_frontier_table(list(frontier))
            print_reached_table(reached_order)
            print("ĐÃ TÌM THẤY GOAL")
            print()

            path = reconstruct_path(current_node)
            print_solution(path)
            return path

        children, next_id = create_children(current_node, next_id)
        added_names = []
        ignored_states = []

        for child in children:
            if child.state not in reached:
                frontier.append(child)
                reached.add(child.state)
                reached_order.append(child.state)
                added_names.append(child.name)
            else:
                ignored_states.append(child.state)

        print_one_bfs_step(
            start=start,
            goal=goal,
            current_node=current_node,
            node_rows=node_rows,
            children=children,
            added_names=added_names,
            ignored_states=ignored_states,
            frontier=frontier,
            reached_order=reached_order
        )

    print("Không tìm thấy goal trong giới hạn bước.")
    return None


# =========================================================
# CHẠY TỰ ĐỘNG
# Muốn đổi đề thì chỉ sửa START ở đây
# =========================================================

START = (1, 2, 3,
         4, 5, 6,
         7, 0, 8)

result = bfs_8_puzzle(start=START, goal=GOAL, max_steps=100)


BFS 8 PUZZLE
START STATE
+---+---+---+
| 1 | 2 | 3 |
+---+---+---+
| 4 | 5 | 6 |
+---+---+---+
| 7 |   | 8 |
+---+---+---+
GOAL STATE
+---+---+---+
| 1 | 2 | 3 |
+---+---+---+
| 4 | 5 | 6 |
+---+---+---+
| 7 | 8 |   |
+---+---+---+
NODE ĐANG XÉT: A
+---+---+---+
| 1 | 2 | 3 |
+---+---+---+
| 4 | 5 | 6 |
+---+---+---+
| 7 |   | 8 |
+---+---+---+
NODE TABLE
----------------------------------------------------------------------------------------------------
Step   Node    Parent    Action    Depth   State
----------------------------------------------------------------------------------------------------
1      A       None      None      0       [1 2 3 4 5 6 7 0 8]

CHILDREN GENERATED
----------------------------------------------------------------------------------------------------
(A, U, 1) -> N1 | [1 2 3 4 0 6 7 5 8] | ADD TO FRONTIER
(A, L, 1) -> N2 | [1 2 3 4 5 6 0 7 8] | ADD TO FRONTIER
(A, R, 1) -> N3 | [1 2 3 4 5 6 7 8 0] | ADD TO FRONTIER

FRONTIER TABLE
-----------------------